# 01 – Thu thập dữ liệu và EDA

Notebook của M1. Phần đầu (mục 1–4) làm task **M1-01** (xác minh nguồn, schema) và **M1-02** (trích xuất dữ liệu thô). Phần sau (mục 5–8) làm **M1-03** (EDA định lượng) và **M1-04** (EDA trực quan) trên chính file thô đó.

**Tải dữ liệu phải chạy trên Google Colab**: cổng dữ liệu Chicago trả lỗi 403 khi truy cập từ máy ở Việt Nam (đã thử lại ngày 18/11/2026).

Mở trên Colab (nhánh của M1, đổi thành `main` sau khi merge):
https://colab.research.google.com/github/tanbentam/KhaiPhaDuLieu/blob/m1/extract-eda/notebooks/01_data_acquisition_eda.ipynb

Trước khi chạy trên Colab:
1. Trên Google Drive có thư mục `KhaiPhaDuLieu/data` (nằm trong My Drive, hoặc là shortcut đặt trong My Drive).
2. Nhánh ghi ở biến `BRANCH` đã được đẩy lên GitHub.

Nếu file thô đã có sẵn trong `data/raw/` thì notebook không tải lại, chỉ đọc và kiểm tra.

In [ ]:
# ===== O THIET LAP (dua tren o dung chung cua notebook 00; them BRANCH va noi Drive) =====
# Colab: clone repo, cai thu vien, noi data/ voi Google Drive.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
BRANCH = "m1/extract-eda"        # doi thanh "main" sau khi nhanh M1 duoc merge
DRIVE_DIR = "KhaiPhaDuLieu"      # = MyDrive/KhaiPhaDuLieu/data
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if IN_COLAB:
    from src.colab import link_drive_data
    print("data/ ->", link_drive_data(DRIVE_DIR))   # file ghi vao data/ se nam luon tren Drive

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## 1. Tham số

Cửa sổ thời gian lấy từ mục `extract` của `config/experiment.yaml`. Giá trị hiện tại (17/04–14/05/2023) là **đề xuất**: bốn tuần trọn từ thứ Hai đến Chủ nhật, không có ngày lễ liên bang, không đổi giờ mùa hè, tuần đầu trùng tuần pilot của M2. Nhóm chốt trước khi tải.

In [ ]:
import json
import pandas as pd
from src.config import load_config, resolve_path
from src.data import extract as E

cfg = load_config()
START, END = cfg["extract"]["start"], cfg["extract"]["end"]
RAW_DIR, MAN_DIR = resolve_path(cfg, "raw_dir"), resolve_path(cfg, "manifests_dir")
NAME = E.raw_name(START, END)
RAW_FILE = RAW_DIR / f"{NAME}.parquet"
MAN_FILE = MAN_DIR / f"{NAME}_manifest.json"
print("Cua so  :", START, "..", END)
print("File tho:", RAW_FILE, "| da co:", RAW_FILE.exists())

## 2. Xác minh nguồn và schema (M1-01)

Đọc metadata thật của cổng dữ liệu và đối chiếu 17 cột cần lấy. Thiếu cột nào thì dừng và báo tên cột. Ô này cần gọi được cổng dữ liệu, nên chỉ chạy trên Colab.

In [ ]:
if RAW_FILE.exists():
    print("File tho da co -> doc schema da luu luc tai.")
    snap = json.loads((MAN_DIR / "schema_snapshot.json").read_text(encoding="utf-8"))
elif IN_COLAB:
    meta = E.fetch_metadata()
    E.check_columns(meta)
    snap = E.schema_snapshot(meta)
    print("Bo du lieu:", snap["name"], "|", snap["n_columns"], "cot | nguon cap nhat:", snap["rows_updated_at_utc"])
else:
    snap = None
    print("May ca nhan va chua co file tho. Chay notebook nay tren Colab, roi tai", RAW_FILE.name,
          "va thu muc manifests tu Drive ve data/.")

if snap:
    dd = E.data_dictionary(snap)
    display(dd[dd.used_by_m1][["field_name", "internal_name", "source_type", "unit", "role"]])

## 3. Đếm và tải dữ liệu thô (M1-02)

`build_raw_extract` đếm số dòng trên server, tải theo trang 50.000 dòng (sắp theo `trip_id`), rồi đối chiếu số dòng. Mỗi trang được lưu thành một file tạm, nên nếu Colab bị ngắt thì chạy lại ô này sẽ tải tiếp từ chỗ dừng. Số dòng lệch với server thì hàm báo lỗi và không ghi file.

File thô là bất biến: giữ nguyên giá trị như API trả về, không lọc dòng nào, và không bị ghi đè nếu đã tồn tại.

In [ ]:
if RAW_FILE.exists():
    man = json.loads(MAN_FILE.read_text(encoding="utf-8"))
    print("Da co file tho, khong tai lai.")
elif IN_COLAB:
    man = E.build_raw_extract(START, END, RAW_DIR, MAN_DIR)
else:
    man = None

if man:
    print("Server dem :", f"{man['rows_server_count']:,}")
    print("Tai ve     :", f"{man['rows_raw']:,}")
    print("Dung luong :", f"{man['file_size_bytes'] / 1e6:.1f} MB")
    print("sha256     :", man["sha256"])

## 4. Kiểm tra nhanh file thô

Kiểm tra file đọc được, khớp manifest và đúng cửa sổ thời gian. Đây chưa phải EDA; các con số dưới đây chỉ để xác nhận việc tải.

In [ ]:
if man:
    raw = pd.read_parquet(RAW_FILE)
    df = E.parse_types(raw)
    assert len(raw) == man["rows_raw"], "so dong khac manifest"
    assert list(raw.columns) == E.COLUMNS, "cot khac danh sach da chon"
    assert E.sha256(RAW_FILE) == man["sha256"], "sha256 khac manifest"
    ts = df["trip_start_timestamp"]
    print("So dong            :", f"{len(df):,}", "|", df.shape[1], "cot")
    print("Thoi gian          :", ts.min(), "..", ts.max(), "|", ts.dt.date.nunique(), "ngay")
    print("Phut trong du lieu :", sorted(int(m) for m in ts.dt.minute.dropna().unique()))
    print("trip_id trung      :", int(raw["trip_id"].duplicated().sum()))
    xy = df[["pickup_centroid_latitude", "pickup_centroid_longitude"]]
    print("Thieu toa do pickup:", f"{xy.isna().any(axis=1).mean():.2%}")
    print("Toa do pickup khac nhau:", f"{len(xy.dropna().drop_duplicates()):,}")
    display(raw.isna().mean().mul(100).round(2).rename("ty le thieu (%)").to_frame())
    display(df.groupby(ts.dt.date).size().rename("so chuyen").to_frame().T)

## 5. EDA định lượng (M1-03)

Từ đây trở xuống chỉ **đếm và thống kê** trên file thô, không loại dòng nào. Việc loại dòng nằm ở notebook 02 và phải có cleaning log.

Các hàm nằm trong `src/data/eda.py`. Ba điều cần nhớ khi đọc số:
- Toạ độ là **tâm census tract / community area**, không phải GPS.
- Thời gian được **làm tròn 15 phút**.
- Số lượt đón là **nhu cầu taxi**, không phải mức độ ùn tắc.

In [ ]:
from src.data import eda as A

METRICS_DIR = ROOT / "outputs" / "metrics"            # eda_summary.csv + thu muc eda/
FIG_DIR = resolve_path(cfg, "figures_dir") / "eda"    # 10 hinh PNG
NOTE = (f"Nguồn: Chicago Data Portal, Taxi Trips (wrvz-psew), {START} đến {END}. "
        "Toạ độ là tâm vùng; thời gian làm tròn 15 phút.")

if man:
    d = A.add_time_parts(df, cfg["time_slices"])      # them gio, thu, lat cat, toc do; khong mat dong
    assert len(d) == len(df)
    display(pd.DataFrame(A.overview(d, raw, man["file_size_bytes"], START, END),
                         columns=["nhom", "chi_so", "gia_tri"]))

### 5.1. Chất lượng: thiếu, trùng, giá trị không hợp lệ

Bảng thứ nhất là tỷ lệ thiếu của từng cột. Bảng thứ hai đếm từng loại vấn đề; cột `action` là cách xử lý **dự kiến** ở notebook 02 (`loai` = bỏ dòng, `danh dau` = giữ và gắn cờ, `giu` = giữ nguyên). Một dòng có thể dính nhiều quy tắc nên không cộng dồn các tỷ lệ.

In [ ]:
if man:
    display(A.missing_table(raw))
    display(A.quality_table(d, START, END))

### 5.2. Phân phối thời lượng, quãng đường, cước

Phân phối chuyến đi lệch phải (ít chuyến rất dài kéo trung bình lên), nên đọc **trung vị, IQR, p90/p95** thay vì chỉ trung bình. `speed_mph` = quãng đường / thời lượng, chỉ tính khi thời lượng > 0.

In [ ]:
if man:
    display(A.numeric_stats(d))

### 5.3. Thời gian: theo ngày, thứ, giờ, lát cắt

Bảng theo ngày so mỗi ngày với trung vị của các ngày cùng thứ; `flag = True` là lệch quá 15%. Ngày bị đánh dấu cần được kiểm chứng (lễ, thời tiết, sự kiện) trước khi kết luận.

In [ ]:
if man:
    by_date = A.daily_anomalies(A.counts_by_date(d))
    print("Ngay thieu trong cua so:", A.missing_dates(d, START, END) or "khong co")
    display(by_date[by_date["flag"]])
    display(A.counts_by_weekday(d))
    display(A.counts_by_slice(d))
    display(A.hour_weekday_matrix(d))

### 5.4. Không gian: toạ độ đón bị lặp nhiều đến mức nào

Đây là phần M2 và M3 cần nhất: số toạ độ khác nhau (tổng, theo lát cắt, theo tuần), tỷ trọng của các toạ độ đông nhất và số chuyến mỗi toạ độ. M2 dùng để đọc đồ thị k-distance của DBSCAN, M3 dùng để chọn `min_cluster_size` cho HDBSCAN.

In [ ]:
if man:
    cc = A.coord_counts(d)
    display(pd.Series(A.coord_concentration(cc), name="gia_tri").to_frame())
    display(cc.head(10))
    display(A.coords_by_group(d, "time_slice"))
    display(A.coords_by_group(d.assign(week_start=A.week_start(d)), "week_start"))
    display(A.counts_by_area(d).head(10))

### 5.5. Ghi bảng ra file

- `outputs/metrics/eda_summary.csv`: mọi chỉ số chính ở dạng (group, metric, value). **Số trong báo cáo lấy từ file này.**
- `outputs/metrics/eda/*.csv`: 12 bảng chi tiết.
- `outputs/metrics/eda/data_quality_report.md`: báo cáo chất lượng, chỉ gồm số liệu.

In [ ]:
if man:
    tables = A.write_tables(d, raw, METRICS_DIR, man["file_size_bytes"], START, END)
    report = A.write_quality_report(d, raw, METRICS_DIR / "eda" / "data_quality_report.md", START, END, NOTE)
    for name, path in {**tables, "data_quality_report": report}.items():
        print(f"{name:32s} {path.relative_to(ROOT)}")

## 6. EDA trực quan (M1-04)

Mười hình, lưu ở `outputs/figures/eda/`. Mỗi hình có tiêu đề, đơn vị, cỡ mẫu và nguồn ở chân hình.

| # | Hình | Để trả lời |
|---|---|---|
| 01, 02 | Histogram thời lượng, quãng đường (cắt trục tại p99) | Đuôi dài, giá trị 0 |
| 03 | Số chuyến theo giờ, ngày thường và cuối tuần | Giờ cao điểm, thấp điểm |
| 04 | Số chuyến theo thứ | Ngày thường so với cuối tuần |
| 05 | Heatmap giờ × thứ | Tổ hợp giờ/ngày đông nhất |
| 06 | Bản đồ điểm đón (1 chấm = 1 toạ độ) | Cấu trúc không gian sơ bộ |
| 07 | Điểm đón và điểm trả, cùng khung và cùng thang màu | Vùng đón có khác vùng trả không |
| 08 | Boxplot thang log | Biến lệch phải |
| 09 | Số chuyến mỗi ngày | Ngày bất thường |
| 10 | Tỷ trọng cộng dồn theo toạ độ | Mức dồn chuyến vào ít toạ độ |

Trong notebook, các hình được thu nhỏ và xếp thành lưới hai cột theo chủ đề để so sánh cạnh nhau; bấm vào hình để mở bản đầy đủ. File PNG trong `outputs/figures/eda/` vẫn giữ kích thước gốc để đưa vào báo cáo.

Hình 06 và 07 vẽ theo toạ độ khác nhau chứ không theo từng chuyến: hàng trăm nghìn chuyến nằm chồng lên vài trăm điểm, nên vẽ từng chuyến sẽ không thấy mật độ.

In [ ]:
from IPython.display import HTML

if man:
    figs = A.write_figures(d, FIG_DIR, NOTE, cfg["time_slices"])     # 10 file PNG day du, dung cho bao cao
    display(HTML(A.figure_grid_html(figs)))                          # hien thi thu gon: luoi 2 cot theo chu de

## 7. Nhận xét từ EDA

<div style="font-size: 85%; line-height: 1.45">

Cửa sổ 17/04–14/05/2023, 554.794 chuyến. Số liệu lấy từ `outputs/metrics/eda_summary.csv` và các bảng trong `outputs/metrics/eda/`. Các nhận xét nói về **nhu cầu đón taxi**, không nói về ùn tắc.

**1. Chuyến đi dồn vào rất ít toạ độ.** 535.167 chuyến có toạ độ chỉ nằm trên 304 điểm (227 tâm census tract, 77 tâm community area); 7 điểm chiếm 50% số chuyến, trong khi 145 điểm có dưới 10 chuyến (Hình 06, 10).
→ Phân cụm thực chất là gom khoảng 300 điểm có trọng số rất chênh nhau. `min_samples` của DBSCAN nên đặt theo tỷ lệ số chuyến, và không diễn giải vị trí cụm chính xác hơn mức tract/khu.

**2. Có hai vùng nhu cầu tách rời: sân bay O'Hare và khu trung tâm.** Khu 76 (O'Hare) chiếm 22,2% số chuyến chỉ với hai toạ độ; ba khu trung tâm 8, 32, 28 chiếm 45,6%; bốn khu cộng lại 67,8% (Hình 06).
→ Kỳ vọng một cụm sân bay tách biệt và một vùng trung tâm đậm đặc. K-Means buộc mọi điểm vào cụm nên dễ bị hai khối này kéo lệch tâm.

**3. Nhu cầu có chu kỳ rõ, đỉnh ở buổi chiều.** Cuối tuần thấp hơn ngày thường 28% (15.600 so với 21.500 chuyến mỗi ngày). Ngày thường đạt đỉnh lúc 15–17 giờ; khung chiều 16–18 giờ cao hơn khung sáng 7–9 giờ 41% (Hình 03, 04, 05).
→ Tách lát cắt thời gian là hợp lý. WEEKDAY_AM nhỏ nhất (60.498 chuyến có toạ độ, 11,3%), cần kiểm tra cỡ mẫu khi lấy mẫu chung.

**4. Điểm đón và điểm trả không đối xứng.** O'Hare chiếm 23,0% lượt đón nhưng chỉ 7,7% lượt trả; điểm trả phân tán hơn (397 toạ độ so với 304). Quãng đường có hai đỉnh: 13,2% số chuyến dài 16–19 dặm, và 96% trong đó có một đầu ở O'Hare (Hình 02, 07).
→ Cụm tìm được mô tả nơi phát sinh nhu cầu, không phải nơi đến. Thuộc tính chuyến đi khác hẳn giữa hai nhóm, nên baseline chỉ dùng `[x_m, y_m]`.

**5. Dữ liệu sạch về định danh, thời gian, toạ độ.** Không trùng `trip_id`, không thiếu ngày, không có điểm ngoài khung Chicago. Chỉ loại 3,54% dòng do thiếu toạ độ đón, còn lại 535.167 dòng. Riêng 10,4% số chuyến có quãng đường bằng 0, dao động 2,2–25,7% tuỳ hãng, nhiều khả năng do thiết bị ghi thiếu (bảng `quality_checks.csv`).
→ Các dòng này được gắn cờ chứ không loại, vì quãng đường không nằm trong ma trận đặc trưng.

**6. Cuối tuần 13–14/05 thấp bất thường, chưa rõ nguyên nhân.** Thấp hơn 20% so với trung bình ba cuối tuần trước, và giảm không đều: khu 33 giảm 73%, Loop 34%, O'Hare 17% (Hình 09).
→ Giả thuyết là Ngày của Mẹ (14/05) hoặc các tuần trước có sự kiện lớn ở khu 33; **chưa kiểm chứng**. M3 cần lưu ý vì lát cắt WEEKEND chứa hai ngày này.

**Giới hạn:** toạ độ là tâm vùng và thời gian làm tròn 15 phút, nên không kết luận ở mức tuyến phố hay phút; bốn tuần mùa xuân 2023 không đại diện cả năm; dữ liệu chỉ gồm taxi truyền thống.

</div>

## 8. Chép kết quả về Drive (chỉ khi chạy trên Colab)

Thư mục `outputs/` trên Colab mất khi đóng phiên, nên chép bảng và hình sang `KhaiPhaDuLieu/data/eda_outputs` trên Drive. Tải thư mục đó về máy, đặt vào `outputs/` của repo rồi mới commit. Chạy trên máy cá nhân thì kết quả đã nằm sẵn trong repo, ô này không làm gì.

In [ ]:
import shutil

if man and IN_COLAB:
    dst = RAW_DIR.parent / "eda_outputs"
    shutil.copytree(FIG_DIR, dst / "figures" / "eda", dirs_exist_ok=True)
    shutil.copytree(METRICS_DIR / "eda", dst / "metrics" / "eda", dirs_exist_ok=True)
    shutil.copy2(METRICS_DIR / "eda_summary.csv", dst / "metrics" / "eda_summary.csv")
    print("Da chep ve Drive:", dst)

## 9. Tiếp theo

Làm sạch (có cleaning log), chiếu EPSG:26916, tạo đặc trưng thời gian, lấy mẫu phân tầng và phát hành `model_input` nằm ở notebook 02.